In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

read in the final features data

In [33]:
company_data = pd.read_csv('./selected_features_data/company_data_ff.csv')
company_data

,ownership_status,United States Sales,United States Engineering,country,city,gpt_sector,country_multiplier,sales_ratio,sales_ratio_adj,adj_hc_ratio,...,employee_highlight_Top Web3 Experience,employee_highlight_HBCU Alum,employee_highlight_$45M Club,employee_highlight_$25M Club,employee_highlight_$35M Club,employee_highlight_$30M Club,employee_highlight_$20M Club,employee_highlight_Jack of All Trades,employee_highlight_$10M Club,num_unique_vc_backers
0,PRIVATE,0,0,United States,New York,NaN,1.0,0.062500,0.062500,1.000000,...,0,0,0,0,0,0,0,0,0,4
1,PRIVATE,0,1,United States,San Francisco,NaN,1.0,0.000000,0.000000,0.990000,...,0,0,0,0,0,0,0,0,0,7
2,PRIVATE,2,2,United States,New York,NaN,1.0,0.100000,0.100000,1.000000,...,0,0,0,0,0,0,1,0,0,2
3,PRIVATE,0,0,United States,San Francisco,NaN,1.0,0.090090,NaN,0.850000,...,0,0,0,0,0,0,0,0,0,12
4,PRIVATE,0,1,United States,San Francisco,Infrastructure,1.0,0.074074,0.071429,0.964286,...,0,0,0,0,0,1,0,0,0,23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14830,PRIVATE,0,0,United States,Nashville,NaN,1.0,0.000000,0.000000,1.000000,...,0,0,0,0,0,0,0,0,0,0
14831,ACQUIRED_OR_MERGED,0,0,United States,Provo,NaN,1.0,0.090090,NaN,0.850000,...,0,0,0,0,0,0,0,0,0,2
14832,PRIVATE,0,0,United States,Raleigh,NaN,1.0,0.090090,NaN,0.850000,...,0,0,0,0,0,0,0,0,0,0
14833,PRIVATE,0,1,United States,San Francisco,NaN,1.0,0.090090,NaN,0.850000,...,0,0,0,0,0,0,0,0,0,5


In [34]:
target_company_data = pd.read_csv('./selected_features_data/target_company_data_ff.csv')
target_company_data

,ownership_status,United States Sales,United States Engineering,country,city,gpt_sector,country_multiplier,sales_ratio,sales_ratio_adj,adj_hc_ratio,...,employee_highlight_Top Web3 Experience,employee_highlight_HBCU Alum,employee_highlight_$45M Club,employee_highlight_$25M Club,employee_highlight_$35M Club,employee_highlight_$30M Club,employee_highlight_$20M Club,employee_highlight_Jack of All Trades,employee_highlight_$10M Club,num_unique_vc_backers
0,PRIVATE,4,23,United States,New York,NaN,1.0,0.163399,0.161290,0.987097,...,0,0,0,0,0,0,0,0,0,18
1,PRIVATE,1,1,United States,New York,Vertical Software,1.0,0.090090,NaN,0.850000,...,0,0,0,0,0,0,0,0,0,10
2,PRIVATE,0,1,United States,New York,NaN,1.0,0.000000,0.000000,0.848148,...,0,0,0,0,0,0,0,0,0,18
3,PRIVATE,1,3,United States,San Francisco,Data,1.0,0.090090,NaN,0.850000,...,1,0,0,0,0,0,0,0,0,22
4,PRIVATE,1,1,United States,New York,Healthcare Software,1.0,0.090090,NaN,0.850000,...,0,1,1,0,0,0,0,0,0,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,PRIVATE,5,2,United States,New York,Vertical Software,1.0,0.000000,0.000000,1.000000,...,1,0,0,0,0,0,0,0,0,10
57,PRIVATE,0,1,United States,San Francisco,NaN,1.0,0.090090,NaN,0.850000,...,0,0,0,0,0,0,0,0,0,14
58,PRIVATE,2,0,United States,Los Angeles,Infrastructure,1.0,0.211268,0.200000,0.946667,...,0,0,0,0,0,0,0,0,0,8
59,PRIVATE,2,4,United States,San Francisco,Horizontal Software,1.0,0.000000,0.000000,0.929412,...,1,0,0,0,0,0,0,0,0,19


## Handle missing values
- analyze missing value stats between company data and target company data

Because there is a huge imbalance between the target (positive) group compared to the rest of the data, we have to keep in mind that the missing% is not equivalent between the target group and the rest of the data. However the difference in their missing% can tell us if a feature could be an indicator of a target company. We do have to keep in mind not to blindly use a feature just because its present in the target companies but not in the rest of the data because that can cause overfitting.

In [35]:
unique_dtypes = company_data.dtypes.unique()
unique_dtypes

array([dtype('O'), dtype('int64'), dtype('float64')], dtype=object)

In [36]:
def clean_missing_data(df):
    # print(df.select_dtypes(include=['string']).columns)
    # print(df.select_dtypes(include=['object']).columns)

    for col in df.select_dtypes(include=['O']).columns:
        # replace empty strings with NaN
        df.loc[:, col] = df[col].str.strip()
        df.loc[:, col] = df[col].replace('', np.nan)

    return df

In [37]:
company_data = clean_missing_data(company_data)
target_company_data = clean_missing_data(target_company_data)

In [38]:
# calculate missing % for each group
missing_target = target_company_data.isna().mean() * 100
missing_rest = company_data.isna().mean() * 100

# combine into one table for comparison
missing_comparison = pd.DataFrame({
    'missing_pct_target': missing_target,
    'missing_pct_rest': missing_rest,
    'missing_pct_diff': (missing_target - missing_rest).abs()
}).sort_values(by='missing_pct_diff', ascending=False)

missing_comparison

,missing_pct_target,missing_pct_rest,missing_pct_diff
ceo_country,16.393443,62.615436,46.221994
revenue_funding_ratio,65.573770,94.122009,28.548238
headcount_growth_12m,27.868852,0.000000,27.868852
adj_emp_delta_12m,27.868852,0.000000,27.868852
emp_delta_12m,27.868852,0.000000,27.868852
...,...,...,...
Prior VC Backed Founder_count,0.000000,0.000000,0.000000
hc_adds_since_funding,0.000000,0.000000,0.000000
years_since_founding,0.000000,0.000000,0.000000
months_since_founding,0.000000,0.000000,0.000000


In [40]:
good_features = missing_comparison[(missing_comparison['missing_pct_diff'] < 30) & (missing_comparison['missing_pct_target'] < 70) & (missing_comparison['missing_pct_rest'] < 70)].index.tolist()
len(good_features)

100

In [42]:
target_company_data = target_company_data[good_features]
company_data = company_data[good_features]

## Handle constant features

drop features that are mostly constant (>99% same value across rows)
- no variance → the model (or any analysis) can't learn anything from a feature that’s always the same.
- extra noise → keeping it would just add unnecessary complexity.

In [43]:
def find_constant_columns(df):
    mostly_constant_cols = []

    for col in df.columns:
        if df[col].nunique(dropna=False) <= 2:
            mostly_constant_cols.append(col)

    print(f"Mostly constant columns: {mostly_constant_cols}")

find_constant_columns(target_company_data)
find_constant_columns(company_data)

Mostly constant columns: ['ownership_status', 'country', 'linkedin_current_peak_count_ratio', 'linkedin_follower_peak_month', 'employee_highlight_$15M Club', 'multi_continent_presence', 'employee_highlight_$20M Club', 'employee_highlight_$30M Club', 'employee_highlight_$35M Club', 'employee_highlight_$25M Club', 'employee_highlight_$45M Club', 'investor_3', 'dominant_function', 'investor_2', 'country_multiplier']
Mostly constant columns: ['multi_continent_presence', 'investor_3']


In [44]:
print('Unique values of investor_3 in target data: ', target_company_data['investor_3'].unique())
print('Unique values of investor_3 in company data: ', company_data['investor_3'].unique())

company_data = company_data.drop(columns=['investor_3'])
target_company_data = target_company_data.drop(columns=['investor_3'])

Unique values of investor_3 in target data:  [0]
Unique values of investor_3 in company data:  [0]


Because both datasets have only the unique value (0) for the feature investor3, it will be dropped. multi_continent_presence will be kept because its 0 or 1 for whether a company has global presence. As for the others I will keep because its probably due to chance that those features happen to nunique <= 2 in the target dataset. Needs further analysis before dropping

## Impute missing values

In [45]:
from sklearn.impute import SimpleImputer

def impute_missing_values(df):
    # Separate numeric and categorical columns
    num_cols = df.select_dtypes(include=["float64", "int64"]).columns
    cat_cols = df.select_dtypes(include=["object"]).columns

    # Define imputers
    num_imputer = SimpleImputer(strategy="median")
    cat_imputer = SimpleImputer(strategy="most_frequent")

    # Fit and transform
    df[num_cols] = num_imputer.fit_transform(df[num_cols])
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

    return df

In [47]:
target_company_data = impute_missing_values(target_company_data)
company_data = impute_missing_values(company_data)

check if missing values are properly handled

In [48]:
print(target_company_data.isnull().sum().sort_values(ascending=False))

headcount_growth_12m                 0
headcount_growth_6m                  0
headcount_0m                         0
emp_highlight_count                  0
Elite Industry Experience_count      0
                                    ..
linkedin_current_peak_count_ratio    0
country                              0
city                                 0
years_since_funding                  0
num_unique_vc_backers                0
Length: 99, dtype: int64


In [49]:
print(company_data.isnull().sum().sort_values(ascending=False))

headcount_growth_12m                 0
headcount_growth_6m                  0
headcount_0m                         0
emp_highlight_count                  0
Elite Industry Experience_count      0
                                    ..
linkedin_current_peak_count_ratio    0
country                              0
city                                 0
years_since_funding                  0
num_unique_vc_backers                0
Length: 99, dtype: int64


## Compare feature distributions between the two groups:
- p-value ≈ probability that the two groups are NOT different by chance.

- Small p-value (e.g., < 0.02) 
    - → Unlikely that the difference between the two groups happened randomly →
    - → The feature behaves differently between target and non-target →
    - → Useful for the model to tell them apart!

In [56]:
from scipy.stats import mannwhitneyu, chi2_contingency

def statistical_feature_test(df_target, df_rest, feature, threshold=0.02):
    try:
        # Numerical feature
        target_values = df_target[feature].dropna()
        rest_values = df_rest[feature].dropna()

        if np.issubdtype(target_values.dtype, np.number):
            stat, p_value = mannwhitneyu(target_values, rest_values, alternative='two-sided')
        else:
            # Categorical feature
            contingency_table = pd.crosstab(df_target[feature], df_rest[feature])
            stat, p_value, _, _ = chi2_contingency(contingency_table)
        
        return p_value < threshold
    except Exception as e:
        print(f"Skipping {feature} due to error: {e}")
        return False

In [58]:
# for feature in company_data.columns:
#     print(feature, ' is not occuring due to random chance: ', statistical_feature_test(target_company_data, company_data, feature))

good_features = [f for f in company_data.columns if statistical_feature_test(target_company_data, company_data, f)]

In [60]:
len(good_features)

63

## Feature importance 

In [67]:
from sklearn.preprocessing import LabelEncoder

def label_encode_dataframe(X):
    X_encoded = X.copy()
    for col in X_encoded.columns:
        if X_encoded[col].dtype == 'object':
            le = LabelEncoder()
            X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
    return X_encoded

In [ ]:
target_company_data['target'] = 1
company_data['target'] = 0
X_train = pd.concat([target_company_data.drop(columns=['target']), company_data.drop(columns=['target'])])
y_train = pd.concat([target_company_data['target'], company_data['target']])

In [68]:
# Before passing through the random forest for feature importance, we need to label encode the categorical features
X_train = label_encode_dataframe(X_train)

In [71]:
from sklearn.ensemble import RandomForestClassifier

def feature_importance_selection(X, y, top_k=20):
    clf = RandomForestClassifier(class_weight='balanced', random_state=42)
    clf.fit(X, y)
    feature_importances = pd.Series(clf.feature_importances_, index=X.columns)
    top_features = feature_importances.sort_values(ascending=False).head(top_k).index.tolist()
    return top_features

In [69]:
top_features = feature_importance_selection(X_train, y_train)

In [70]:
top_features

['current_max_ratio',
 'linkedin_follower_growth_1m',
 'trailing_growth_max',
 'headcount_volatility_score',
 'trailing_growth_avg',
 'trailing_growth_std',
 'emp_highlight_count',
 'twitter_follower_count_volatility',
 'twitter_follower_trend',
 'linkedin_follower_growth_6m',
 'linkedin_follower_growth_3m',
 'employee_highlight_Major Tech Company Experience',
 'employee_highlight_Top University',
 'investor_1',
 'linkedin_follower_growth_12m',
 'followers_per_linkedin_employee',
 'linkedin_follower_count_volatility',
 'years_since_funding',
 'employee_highlight_Top Company Alum',
 'headcount_growth_1m']

## Quick verification of feature choice

In [72]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

def quick_validation(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

    clf = RandomForestClassifier(class_weight='balanced', random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    print(classification_report(y_test, y_pred, digits=4))

X_selected = X_train[top_features]
quick_validation(X_selected, y_train)

              precision    recall  f1-score   support

           0     0.9989    1.0000    0.9994      4451
           1     1.0000    0.7222    0.8387        18

    accuracy                         0.9989      4469
   macro avg     0.9994    0.8611    0.9191      4469
weighted avg     0.9989    0.9989    0.9988      4469

